In [2]:
import pandas as pd
import requests
import json
import random
import re
import pandas as pd
import ast
import re
from collections import defaultdict

In [6]:



def find_all_occurrences(text, substring):
    return [(m.start(), m.end()) for m in re.finditer(re.escape(substring), text)]


def map_llm_spans_to_offsets(df, pred_col="predicted_spans"):
    pred_spans_by_article = {}

    for _, row in df.iterrows():
        article_id = str(row["article_id"])
        text = str(row["article_text"])

        preds = row[pred_col]
        if isinstance(preds, str):
            import ast
            preds = ast.literal_eval(preds)

        used = set()
        offsets = []

        for span_text in preds:
            span_text = span_text.strip(' "\'.,')

            if not span_text:
                continue

            matches = list(re.finditer(re.escape(span_text), text))

            for m in matches:
                span = (m.start(), m.end())
                if span not in used:
                    offsets.append(span)
                    used.add(span)
                    break

        # 🔥 FIX: merge overlaps here
        pred_spans_by_article[article_id] = merge_overlapping_spans(offsets)

    return pred_spans_by_article

    return pred_spans_by_article
def save_as_labels(pred_spans_by_article, output_file):
    with open(output_file, "w", encoding="utf-8") as f:
        for article_id in sorted(pred_spans_by_article.keys(), key=str):
            spans = pred_spans_by_article[article_id]

            for start, end in sorted(spans):
                if start < end:
                    f.write(f"{article_id}\t{start}\t{end}\n")
def merge_overlapping_spans(spans):
    if not spans:
        return []

    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    merged = [spans[0]]

    for start, end in spans[1:]:
        last_start, last_end = merged[-1]

        if start <= last_end:  # overlap
            merged[-1] = (last_start, max(last_end, end))
        else:
            merged.append((start, end))

    return merged

In [7]:
from span_eval_helper_updated import run_official_si_scorer


def score_llm_predictions(
    csv_file,
    gold_labels_file,
    scorer_script_path,
    submission_output_file="llm_predictions.labels"
):
    df = pd.read_csv(csv_file)

    pred_spans_by_article = map_llm_spans_to_offsets(df)

    save_as_labels(pred_spans_by_article, submission_output_file)

    result = run_official_si_scorer(
        scorer_script_path=scorer_script_path,
        submission_file=submission_output_file,
        gold_labels_file=gold_labels_file,
    )

    return result

In [10]:
result = score_llm_predictions(
    csv_file="val_results.csv",
    gold_labels_file="PTC/dev-task-SI.labels",
    scorer_script_path="task-SI_scorer.py",
    submission_output_file="llm_predictions.labels"
)

print("Submission file:", result["command"])
print(result["stdout"])
print(result["stderr"])

Submission file: ['python', 'task-SI_scorer.py', '-s', 'llm_predictions.labels', '-r', 'PTC/dev-task-SI.labels']
2026-04-17 12:42:38,535 - INFO - Checking user submitted file
2026-04-17 12:42:38,550 - INFO - Scoring the submission with precision and recall method
2026-04-17 12:42:38,700 - INFO - Precision=296.293196/930=0.318595	Recall=468.509166/940=0.498414
2026-04-17 12:42:38,700 - INFO - F1=0.388716


